# 외부 데이터 수집·전처리: 2026년 6월 기준

이 노트북은 공모전 소비 원자료와 결합 가능한 외부 데이터를 **노트북 안에서** 검증·정규화합니다. 공급지표는 업종 교차표가 검토되기 전까지 만들지 않습니다.

- 인구 노출량: 행정안전부 주민등록 인구통계
- 사업체 후보: 국세청 100대 생활업종 및 소상공인시장진흥공단 상가(상권)정보
- 기준시점: 2026년 6월 (소비자료 2026년 1–6월과 맞춤)

In [1]:
from pathlib import Path
from zipfile import ZipFile

import polars as pl

ROOT = Path("..") if (Path("..") / "ABP_CONTEST_DATA.csv").exists() else Path(".")
RAW_DIR = ROOT / "data" / "external" / "raw"
PROCESSED_DIR = ROOT / "data" / "external" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

ABP_PATH = ROOT / "ABP_CONTEST_DATA.csv"
MOIS_PATH = RAW_DIR / "mois_age_population_sgg_202606.csv"
NTS_PATH = RAW_DIR / "nts_100_living_industries_20260630.csv"
SBIZ_PATH = RAW_DIR / "sbiz_stores_20260630.zip"

for path in (ABP_PATH, MOIS_PATH, NTS_PATH, SBIZ_PATH):
    assert path.exists(), f"원본 파일이 없습니다: {path}"

print(f"프로젝트 경로: {ROOT.resolve()}")
print(f"산출 경로: {PROCESSED_DIR.resolve()}")

프로젝트 경로: /home/user/finance-modeling
산출 경로: /home/user/finance-modeling/data/external/processed


## 출처와 사용 범위

| 데이터 | 기준일 | 이 노트북의 용도 | 원문 출처 |
|---|---:|---|---|
| 주민등록 연령·성별 인구 | 2026-06 | 내국인 성별·연령 소비의 분모 후보 | [행정안전부 주민등록 인구통계](https://jumin.mois.go.kr/ageStatMonth.do) |
| 국세청 100대 생활업종 | 2026-06-30 | 시군구 사업체 수의 독립적 교차검증 | [공공데이터포털](https://www.data.go.kr/data/15061118/fileData.do) |
| 소상공인 상가(상권)정보 | 2026-06-30 | 업종 분류 카탈로그와 공급지표 후보 | [공공데이터포털](https://www.data.go.kr/data/15083033/fileData.do) |

국세청 자료의 3개 미만 셀은 0으로 제공될 수 있으므로, 0을 실제 부재로 해석하지 않습니다. 상가정보의 분류는 BC카드 11개 업종과 정의가 다르므로 자동 매핑하지 않습니다.

In [2]:
provenance = pl.DataFrame(
    {
        "dataset": ["MOIS age/sex population", "NTS 100 living industries", "SEMAS store information"],
        "as_of": ["2026-06", "2026-06-30", "2026-06-30"],
        "raw_file": [MOIS_PATH.name, NTS_PATH.name, SBIZ_PATH.name],
        "url": [
            "https://jumin.mois.go.kr/ageStatMonth.do",
            "https://www.data.go.kr/data/15061118/fileData.do",
            "https://www.data.go.kr/data/15083033/fileData.do",
        ],
    }
)
provenance

dataset,as_of,raw_file,url
str,str,str,str
"""MOIS age/sex population""","""2026-06""","""mois_age_population_sgg_202606…","""https://jumin.mois.go.kr/ageSt…"
"""NTS 100 living industries""","""2026-06-30""","""nts_100_living_industries_2026…","""https://www.data.go.kr/data/15…"
"""SEMAS store information""","""2026-06-30""","""sbiz_stores_20260630.zip""","""https://www.data.go.kr/data/15…"


## BC카드 지역 키

`시군구명`만으로는 동명이인 지역이 있으므로, 모든 결합은 `시도명 + 시군구명` 복합키로 합니다. 세종은 시도명과 시군구명이 같아 행정안전부 표기와 별도로 맞춥니다.

In [3]:
ABP_REGION_KEY = ["SIDO_NM", "CCG_NM"]
abp_regions = (
    pl.scan_csv(ABP_PATH)
    .select(ABP_REGION_KEY)
    .unique()
    .collect()
    .with_columns(
        pl.when(pl.col("SIDO_NM") == pl.col("CCG_NM"))
        .then(pl.col("SIDO_NM"))
        .otherwise(pl.concat_str([pl.col("SIDO_NM"), pl.col("CCG_NM")], separator=" "))
        .alias("source_region_label")
    )
    .sort(ABP_REGION_KEY)
)
assert abp_regions.height == 255
abp_regions.head()

SIDO_NM,CCG_NM,source_region_label
str,str,str
"""강원특별자치도""","""강릉시""","""강원특별자치도 강릉시"""
"""강원특별자치도""","""고성군""","""강원특별자치도 고성군"""
"""강원특별자치도""","""동해시""","""강원특별자치도 동해시"""
"""강원특별자치도""","""삼척시""","""강원특별자치도 삼척시"""
"""강원특별자치도""","""속초시""","""강원특별자치도 속초시"""


## 1. 성별·연령 인구 정규화

공모전 설명의 연령코드 1은 `20대 이하`로 적혀 있으나 코드 2도 `20대`여서 겹칩니다. 여기서는 코드 1을 **20세 미만**, 코드 2를 **20–29세**로 두는 보수적 임시 규칙을 사용합니다. 제출 전 플랫폼 코드북으로 확인해야 합니다. 외국인·법인에는 이 인구 분모를 적용하지 않습니다.

In [4]:
population_wide = pl.read_csv(MOIS_PATH, encoding="cp949")
month_prefix = "2026년06월"

population_base = population_wide.select(
    pl.col("행정구역").cast(pl.Utf8).str.replace_all(r"\s+\(\d{10}\)", "").str.strip_chars().alias("source_region_label"),
    pl.col("행정구역").cast(pl.Utf8).str.extract(r"\((\d{10})\)").alias("ADMIN_REGION_CODE"),
    pl.all().exclude("행정구역"),
)

# 원본에는 세종의 광역(3600000000)·시군구(3611000000) 집계가 같은 값으로 중복된다.
sejong_rows = population_base.filter(pl.col("source_region_label") == "세종특별자치시")
assert sejong_rows.height == 2
assert sejong_rows.select(pl.all().exclude("ADMIN_REGION_CODE")).unique().height == 1
population_base = population_base.filter(pl.col("ADMIN_REGION_CODE") != "3600000000")
assert population_base.select("source_region_label").n_unique() == population_base.height

age_bands = {
    "1": ["0~9세", "10~19세"],
    "2": ["20~29세"],
    "3": ["30~39세"],
    "4": ["40~49세"],
    "5": ["50~59세"],
    "6": ["60~69세", "70~79세", "80~89세", "90~99세", "100세 이상"],
}
sex_prefix = {"1": "남", "2": "여"}

parts = []
for gender_cd, sex in sex_prefix.items():
    for age_cd, ages in age_bands.items():
        source_columns = [f"{month_prefix}_{sex}_{age}" for age in ages]
        missing = set(source_columns) - set(population_wide.columns)
        assert not missing, f"예상 인구 열이 없습니다: {missing}"
        parts.append(
            population_base.select(
                "source_region_label",
                "ADMIN_REGION_CODE",
                pl.lit(gender_cd).alias("GENDER_CD"),
                pl.lit(age_cd).alias("AGE_CD"),
                pl.sum_horizontal(
                    [pl.col(column).cast(pl.Utf8).str.replace_all(",", "").cast(pl.Int64) for column in source_columns]
                ).alias("POPULATION"),
            )
        )

population_long = pl.concat(parts)
extra_population_regions = population_long.select("source_region_label").unique().join(
    abp_regions.select("source_region_label"), on="source_region_label", how="anti"
)
unmatched_abp_regions = abp_regions.select("source_region_label").join(
    population_long.select("source_region_label").unique(), on="source_region_label", how="anti"
)
assert unmatched_abp_regions.is_empty(), unmatched_abp_regions

population_processed = (
    population_long.join(abp_regions, on="source_region_label", how="inner")
    .select(
        "SIDO_NM", "CCG_NM", "ADMIN_REGION_CODE", "GENDER_CD", "AGE_CD", "POPULATION"
    )
    .sort(["SIDO_NM", "CCG_NM", "GENDER_CD", "AGE_CD"])
)
assert population_processed.height == 255 * 2 * 6
assert population_processed.select(pl.col("POPULATION").min()).item() >= 0

POPULATION_OUTPUT = PROCESSED_DIR / "population_sgg_age_sex_202606.csv"
population_processed.write_csv(POPULATION_OUTPUT)
print(f"인구 분모 {population_processed.height:,}행 저장: {POPULATION_OUTPUT}")
print(f"공모전 지역 외 행정안전부 집계행 제외: {extra_population_regions.height}개")
population_processed.head()

인구 분모 3,060행 저장: ../data/external/processed/population_sgg_age_sex_202606.csv
공모전 지역 외 행정안전부 집계행 제외: 40개


SIDO_NM,CCG_NM,ADMIN_REGION_CODE,GENDER_CD,AGE_CD,POPULATION
str,str,str,str,str,i64
"""강원특별자치도""","""강릉시""","""5115000000""","""1""","""1""",13692
"""강원특별자치도""","""강릉시""","""5115000000""","""1""","""2""",11004
"""강원특별자치도""","""강릉시""","""5115000000""","""1""","""3""",11101
"""강원특별자치도""","""강릉시""","""5115000000""","""1""","""4""",13475
"""강원특별자치도""","""강릉시""","""5115000000""","""1""","""5""",18427


## 2. 국세청 100대 생활업종: 독립 교차검증 자료

이 자료는 시군구·업종별 사업체 수를 보완하지만, BC카드 업종과 명칭 및 포괄 범위가 달라 공급분모에 바로 넣지 않습니다. 먼저 원본 열을 정리하고 지역 커버리지를 확인합니다.

In [5]:
import sys

nts_raw = pl.read_csv(NTS_PATH)
COUNTS = ["NTS_CURRENT_COUNT", "NTS_PREVIOUS_COUNT", "NTS_YEAR_AGO_COUNT"]
nts = nts_raw.rename(
    {
        "업종": "NTS_INDUSTRY",
        "시도": "SIDO_NM",
        "시군구": "CCG_NM",
        " 당월 ": "NTS_CURRENT_COUNT",
        " 전월 ": "NTS_PREVIOUS_COUNT",
        " 전년동월 ": "NTS_YEAR_AGO_COUNT",
    }
).select("SIDO_NM", "CCG_NM", "NTS_INDUSTRY", *[pl.col(c).cast(pl.Float64) for c in COUNTS])
assert nts.height == 24_600
assert nts.select(pl.col("NTS_CURRENT_COUNT").min()).item() >= 0
# 화성시 잔여 행은 2026-02 구 신설 전 흔적(당월 0). 전년동월 비교는 화성 4개 구에서 할 수 없다.
nts = nts.filter(~((pl.col("SIDO_NM") == "경기도") & (pl.col("CCG_NM") == "화성시")))
raw_totals = nts.select(pl.col(COUNTS).sum()).row(0)

# 원본(2026-06-30)은 7월 개편 명칭을 쓴다. 공모전(개편 전) 지역으로 되돌린다.
# - 전남광주통합특별시: 구 → 광주광역시, 시·군 → 전라남도 / 세종: 시군구 "0"
# - 인천: 서해구·검단구(+잔여 서구) → 서구, 영종구 → 중구,
#   제물포구 = 옛 중구 내륙 + 옛 동구 → 상가정보 점포의 법정동 비율로 동구 몫을 나눈다.
sys.path.append(str(ROOT / "scripts"))
from fetch_external import DONGGU

with ZipFile(SBIZ_PATH) as archive, archive.open("소상공인시장진흥공단_상가(상권)정보_인천_202606.csv") as source:
    jemulpo_stores = pl.read_csv(source, columns=["시군구코드", "법정동명"], schema_overrides={"시군구코드": pl.Utf8})
DONGGU_SHARE = jemulpo_stores.filter(pl.col("시군구코드") == "28125")["법정동명"].is_in(list(DONGGU)).mean()
print(f"제물포구 점포 중 옛 동구 비율: {DONGGU_SHARE:.3f}")  # ponytail: 전 업종 공통 비율, 업종별 편차가 크면 업종별로

is_jemulpo = (pl.col("SIDO_NM") == "인천광역시") & (pl.col("CCG_NM") == "제물포구")
donggu = nts.filter(is_jemulpo).with_columns(pl.lit("동구").alias("CCG_NM"), *[pl.col(c) * DONGGU_SHARE for c in COUNTS])
rest = nts.with_columns(*[pl.when(is_jemulpo).then(pl.col(c) * (1 - DONGGU_SHARE)).otherwise(pl.col(c)).alias(c) for c in COUNTS])
INCHEON_BACK = {"서해구": "서구", "검단구": "서구", "영종구": "중구", "제물포구": "중구"}
nts = (
    pl.concat([rest, donggu])
    .with_columns(
        pl.when(pl.col("SIDO_NM") == "인천광역시").then(pl.col("CCG_NM").replace(INCHEON_BACK))
        .when(pl.col("SIDO_NM") == "세종특별자치시").then(pl.col("SIDO_NM"))
        .otherwise(pl.col("CCG_NM"))
        .alias("CCG_NM"),
        pl.when(pl.col("SIDO_NM") != "전남광주통합특별시").then(pl.col("SIDO_NM"))
        .when(pl.col("CCG_NM").str.ends_with("구")).then(pl.lit("광주광역시"))
        .otherwise(pl.lit("전라남도"))
        .alias("SIDO_NM"),
    )
    .group_by("SIDO_NM", "CCG_NM", "NTS_INDUSTRY")
    .agg(pl.col(COUNTS).sum())
    .sort(["SIDO_NM", "CCG_NM", "NTS_INDUSTRY"])
)

missing_nts_regions = abp_regions.select(ABP_REGION_KEY).join(nts.select(ABP_REGION_KEY).unique(), on=ABP_REGION_KEY, how="anti")
assert missing_nts_regions.is_empty(), missing_nts_regions
# 사업자 0인 업종은 행 자체가 없으므로 지역당 행 수는 100 미만일 수 있다.
assert nts.join(abp_regions, on=ABP_REGION_KEY, how="anti").is_empty() and nts.select(ABP_REGION_KEY).n_unique() == 255
assert all(abs(a - b) < 1e-6 for a, b in zip(nts.select(pl.col(COUNTS).sum()).row(0), raw_totals)), "사업자 합계 보존 실패"
incheon_seogu = nts.filter((pl.col("SIDO_NM") == "인천광역시") & (pl.col("CCG_NM") == "서구"))["NTS_CURRENT_COUNT"].sum()
assert incheon_seogu > 30_000, f"인천 서구 사업자 {incheon_seogu:,.0f}: 개편 구 합산 실패"

NTS_OUTPUT = PROCESSED_DIR / "nts_100_living_industries_sgg_202606.csv"
nts.write_csv(NTS_OUTPUT)
print(f"국세청 {nts.height:,}행 저장: {NTS_OUTPUT} (인천 서구 사업자 {incheon_seogu:,.0f})")

제물포구 점포 중 옛 동구 비율: 0.406
국세청 24,398행 저장: ../data/external/processed/nts_100_living_industries_sgg_202606.csv (인천 서구 사업자 38,693)


## 3. 소상공인 상가(상권)정보의 업종 카탈로그

전국 상가 원본 전체를 집계하지 않고, 먼저 10/75/247 체계의 고유 업종 조합만 읽어 업종 교차표를 검토할 수 있게 만듭니다. 이 단계는 원본을 변환하거나 지표를 산출하지 않습니다.

In [6]:
CATEGORY_COLUMNS = [
    "상권업종대분류코드",
    "상권업종대분류명",
    "상권업종중분류코드",
    "상권업종중분류명",
    "상권업종소분류코드",
    "상권업종소분류명",
]

with ZipFile(SBIZ_PATH) as archive:
    members = [name for name in archive.namelist() if name.endswith(".csv")]
    catalog_parts = []
    for name in members:
        with archive.open(name) as source:
            catalog_parts.append(pl.read_csv(source, columns=CATEGORY_COLUMNS).unique())

category_catalog = (
    pl.concat(catalog_parts)
    .unique()
    .sort(["상권업종대분류명", "상권업종중분류명", "상권업종소분류명"])
)
CATALOG_OUTPUT = PROCESSED_DIR / "sbiz_industry_catalog_202606.csv"
category_catalog.write_csv(CATALOG_OUTPUT)
print(f"상가정보 업종 카탈로그 {category_catalog.height:,}행 저장: {CATALOG_OUTPUT}")
category_catalog.filter(
    pl.col("상권업종대분류명").is_in(["음식", "소매"])
).head(20)

상가정보 업종 카탈로그 247행 저장: ../data/external/processed/sbiz_industry_catalog_202606.csv


상권업종대분류코드,상권업종대분류명,상권업종중분류코드,상권업종중분류명,상권업종소분류코드,상권업종소분류명
str,str,str,str,str,str
"""G2""","""소매""","""G211""","""가구 소매""","""G21101""","""가구 소매업"""
"""G2""","""소매""","""G208""","""가전·통신 소매""","""G20803""","""가전제품 소매업"""
"""G2""","""소매""","""G208""","""가전·통신 소매""","""G20801""","""컴퓨터/소프트웨어 소매업"""
"""G2""","""소매""","""G208""","""가전·통신 소매""","""G20802""","""핸드폰 소매업"""
"""G2""","""소매""","""G221""","""기타 상품 소매""","""G22199""","""그 외 기타 상품 전문 소매업"""
…,…,…,…,…,…
"""G2""","""소매""","""G209""","""섬유·의복·신발 소매""","""G20909""","""실/섬유제품 소매업"""
"""G2""","""소매""","""G209""","""섬유·의복·신발 소매""","""G20907""","""액세서리/잡화 소매업"""
"""G2""","""소매""","""G209""","""섬유·의복·신발 소매""","""G20902""","""여성 의류 소매업"""


## 4. 공급지표 산출 전 교차표 게이트

`sbiz_crosswalk_approved.csv`에는 `TP_BUZ_NO`, `TP_BUZ_NM`, `상권업종소분류코드`, `mapping_rationale`를 담습니다. BC카드 업종 정의와 소상공인 분류가 일치한다는 근거를 각 행에 남긴 뒤에만 상가 수를 집계합니다. 파일이 없으면 모델 입력을 만들지 않고 멈추는 것이 의도된 동작입니다.

In [7]:
CROSSWALK_PATH = PROCESSED_DIR / "sbiz_crosswalk_approved.csv"
required_crosswalk_columns = {
    "TP_BUZ_NO",
    "TP_BUZ_NM",
    "상권업종소분류코드",
    "mapping_rationale",
}

if CROSSWALK_PATH.exists():
    crosswalk = pl.read_csv(CROSSWALK_PATH)
    missing_columns = required_crosswalk_columns - set(crosswalk.columns)
    assert not missing_columns, f"교차표 필수 열 누락: {missing_columns}"
    assert crosswalk.select("mapping_rationale").drop_nulls().height == crosswalk.height
    print(f"승인된 교차표 {crosswalk.height}행을 확인했습니다. 다음 노트북에서 공급지표를 집계할 수 있습니다.")
else:
    print("승인된 업종 교차표가 없습니다. 공급지표와 MCMC 의사결정 모델은 여기서 대기합니다.")

승인된 업종 교차표가 없습니다. 공급지표와 MCMC 의사결정 모델은 여기서 대기합니다.


## 다음 단계

1. 공모전 코드북으로 `AGE_CD=1`의 범위를 확인합니다.
2. 업종 교차표를 검토·승인합니다.
3. 그 뒤에만 인구 분모와 공급지표를 결합해 음이항 수요모형과 MCMC 의사결정 점수를 적합합니다.